# Zero-Shot NER on QUAERO FrenchMed with Qwen2.5-14B-Instruct

Ce notebook charge un LLM quantifié en 4-bit sur le GPU Colab (T4 15Go),
extrait les entités nommées en zero-shot, et évalue avec les mêmes métriques
que le pipeline GLiNER (exact span match, F1 par label).

Les fichiers `test_gliner.json` et `train_gliner.json` doivent être dans `/content/`.

In [ ]:
# 1. Installation des dépendances
!pip install -q transformers accelerate bitsandbytes matplotlib

In [ ]:
# 2. Vérification GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} Go")

In [ ]:
# 3. Vérification des fichiers
import os

for f in ['test_gliner.json', 'train_gliner.json']:
    path = f'/content/{f}'
    if os.path.exists(path):
        print(f"OK: {path}")
    else:
        print(f"MANQUANT: {path} — uploadez-le dans /content/")

In [ ]:
# 4. Configuration
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"  # Changer en Qwen/Qwen2.5-72B-Instruct pour Colab Pro

QUAERO_LABELS = ["DISO", "PROC", "ANAT", "CHEM", "DEVI", "LIVB", "PHYS", "PHEN", "GEOG", "OBJC"]

SYSTEM_PROMPT = """Tu es un expert en annotation d'entités nommées biomédicales.
Extrais toutes les entités nommées du texte ci-dessous.

Catégories possibles :
- DISO : Maladie ou trouble
- PROC : Procédure médicale
- ANAT : Structure anatomique
- CHEM : Substance chimique ou médicament
- DEVI : Dispositif médical
- LIVB : Être vivant
- PHYS : Processus physiologique
- PHEN : Phénomène
- GEOG : Lieu géographique
- OBJC : Objet

Réponds UNIQUEMENT avec un tableau JSON, sans explication.
Chaque élément : {"text": "...", "label": "..."}

Exemple :
[{"text": "cancer du sein", "label": "DISO"}, {"text": "mammographie", "label": "PROC"}]

Si aucune entité n'est trouvée, réponds : []"""

MAX_DOCS = None  # None = tous, ou un nombre pour tester (ex: 50)

In [ ]:
# 5. Chargement du modèle en 4-bit
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

print(f"Chargement de {MODEL_ID} en 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"Modèle chargé. VRAM utilisée: {torch.cuda.memory_allocated()/1e9:.1f} Go")

In [ ]:
# 6. Fonctions utilitaires
import json
import re
import time
from collections import defaultdict


def generate_response(text, max_new_tokens=2048):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Texte :\n{text}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response


def parse_llm_response(response):
    response = response.strip()
    match = re.search(r'\[.*\]', response, re.DOTALL)
    if not match:
        return []
    try:
        entities = json.loads(match.group())
        if not isinstance(entities, list):
            return []
        return [
            {"text": e["text"], "label": e["label"]}
            for e in entities
            if isinstance(e, dict) and "text" in e and "label" in e and e["label"] in QUAERO_LABELS
        ]
    except json.JSONDecodeError:
        return []


def build_char_offsets(tokens):
    offsets, pos = [], 0
    for tok in tokens:
        offsets.append((pos, pos + len(tok)))
        pos += len(tok) + 1
    return " ".join(tokens), offsets


def find_entity_in_tokens(entity_text, label, text, offsets):
    results = []
    entity_lower = entity_text.lower()
    text_lower = text.lower()
    start = 0
    while True:
        idx = text_lower.find(entity_lower, start)
        if idx == -1:
            break
        end_char = idx + len(entity_text)
        ts = te = None
        for i, (cs, ce) in enumerate(offsets):
            if ts is None and cs <= idx < ce:
                ts = i
            if cs < end_char <= ce:
                te = i
        if ts is not None and te is not None and ts <= te:
            results.append((ts, te, label))
        start = idx + 1
    return results


def evaluate(gold, pred):
    tp = len(gold & pred)
    return tp, len(pred) - tp, len(gold) - tp


def prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return round(p, 4), round(r, 4), round(f1, 4)


print("Fonctions chargées.")

In [ ]:
# 7. Chargement des données
docs = json.loads(open('/content/test_gliner.json', encoding='utf-8').read())
if MAX_DOCS:
    docs = docs[:MAX_DOCS]
print(f"Test set: {len(docs)} documents")

In [ ]:
# 8. Évaluation zero-shot
global_tp = global_fp = global_fn = 0
per_label = defaultdict(lambda: [0, 0, 0])
errors = 0
all_predictions = []

start_time = time.time()

for i, doc in enumerate(docs):
    tokens = doc["tokenized_text"]
    text, offsets = build_char_offsets(tokens)
    gold = {(e[0], e[1], e[2]) for e in doc["ner"]}

    response = generate_response(text)
    entities = parse_llm_response(response)

    pred = set()
    for ent in entities:
        spans = find_entity_in_tokens(ent["text"], ent["label"], text, offsets)
        for span in spans:
            pred.add(span)

    all_predictions.append({"gold": gold, "pred": pred, "raw": response})

    tp, fp, fn = evaluate(gold, pred)
    global_tp += tp
    global_fp += fp
    global_fn += fn

    for label in QUAERO_LABELS:
        g = {s for s in gold if s[2] == label}
        p = {s for s in pred if s[2] == label}
        ltp, lfp, lfn = evaluate(g, p)
        per_label[label][0] += ltp
        per_label[label][1] += lfp
        per_label[label][2] += lfn

    if not entities and gold:
        errors += 1

    if (i + 1) % 50 == 0 or i == 0:
        elapsed = time.time() - start_time
        speed = (i + 1) / elapsed
        eta = (len(docs) - i - 1) / speed if speed > 0 else 0
        p_cur, r_cur, f1_cur = prf(global_tp, global_fp, global_fn)
        print(f"[{i+1}/{len(docs)}] F1={f1_cur:.3f} P={p_cur:.3f} R={r_cur:.3f}"
              f"  ({speed:.1f} doc/s, ETA {eta/60:.0f}min)")

elapsed = time.time() - start_time
p_all, r_all, f1_all = prf(global_tp, global_fp, global_fn)
print(f"\nDone in {elapsed/60:.1f} min")
print(f"F1={f1_all:.3f}  P={p_all:.3f}  R={r_all:.3f}")
print(f"Parse errors: {errors}/{len(docs)}")

In [ ]:
# 9. Résultats par label
print(f"\n{'Label':<8} {'P':>8} {'R':>8} {'F1':>8}  {'TP':>5} {'FP':>5} {'FN':>5}")
print("-" * 55)
for label in sorted(QUAERO_LABELS, key=lambda l: -prf(*per_label[l])[2]):
    ltp, lfp, lfn = per_label[label]
    lp, lr, lf1 = prf(ltp, lfp, lfn)
    print(f"{label:<8} {lp:>8.3f} {lr:>8.3f} {lf1:>8.3f}  {ltp:>5} {lfp:>5} {lfn:>5}")
print("-" * 55)
print(f"{'GLOBAL':<8} {p_all:>8.3f} {r_all:>8.3f} {f1_all:>8.3f}  {global_tp:>5} {global_fp:>5} {global_fn:>5}")

In [ ]:
# 10. Graphiques
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F1 par label
ax = axes[0]
labels_sorted = sorted(QUAERO_LABELS, key=lambda l: -prf(*per_label[l])[2])
f1s = [prf(*per_label[l])[2] for l in labels_sorted]
colors = ["#4CAF50" if f > 0.5 else "#FF9800" if f > 0.2 else "#F44336" for f in f1s]
bars = ax.barh(labels_sorted, f1s, color=colors)
for bar, val in zip(bars, f1s):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9)
ax.set_xlim(0, 1)
ax.set_xlabel("F1")
ax.set_title(f"F1 per label — {MODEL_ID.split('/')[-1]} (zero-shot)")
ax.grid(True, alpha=0.3, axis="x")

# P / R / F1 global
ax = axes[1]
metrics_names = ["Precision", "Recall", "F1"]
vals = [p_all, r_all, f1_all]
colors_m = ["#2196F3", "#FF9800", "#4CAF50"]
bars = ax.bar(metrics_names, vals, color=colors_m, width=0.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", fontweight="bold", fontsize=12)
ax.set_ylim(0, 1)
ax.set_title(f"Global — {MODEL_ID.split('/')[-1]} (zero-shot)\n{len(docs)} docs, {elapsed/60:.0f} min")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("/content/zero_shot_results.png", dpi=150)
plt.show()
print("Plot saved → /content/zero_shot_results.png")

In [ ]:
# 11. Export CSV (téléchargeable)
import csv
from google.colab import files

row = {
    "model": MODEL_ID,
    "method": "zero-shot",
    "n_docs": len(docs),
    "P": p_all, "R": r_all, "F1": f1_all,
    "TP": global_tp, "FP": global_fp, "FN": global_fn,
    "time_min": round(elapsed / 60, 1),
}
for label in QUAERO_LABELS:
    ltp, lfp, lfn = per_label[label]
    _, _, lf1 = prf(ltp, lfp, lfn)
    row[f"F1_{label}"] = lf1

with open("/content/llm_results.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(row.keys()))
    writer.writeheader()
    writer.writerow(row)

print("Results saved → /content/llm_results.csv")
files.download("/content/llm_results.csv")
files.download("/content/zero_shot_results.png")

In [ ]:
# 12. Inspection d'un document (optionnel)
DOC_IDX = 0

doc = docs[DOC_IDX]
tokens = doc["tokenized_text"]
text = " ".join(tokens)
print(f"Text: {text}\n")

gold = all_predictions[DOC_IDX]["gold"]
pred = all_predictions[DOC_IDX]["pred"]

print("=== GOLD ===")
for ts, te, label in sorted(gold):
    marker = "OK" if (ts, te, label) in pred else "MISSED"
    print(f"  [{label}] {ts}-{te}  '{' '.join(tokens[ts:te+1])}'  {marker}")

print("\n=== PREDICTIONS ===")
for ts, te, label in sorted(pred):
    marker = "OK" if (ts, te, label) in gold else "FALSE POS"
    print(f"  [{label}] {ts}-{te}  '{' '.join(tokens[ts:te+1])}'  {marker}")

tp, fp, fn = evaluate(gold, pred)
p, r, f1 = prf(tp, fp, fn)
print(f"\nDoc P={p:.3f}  R={r:.3f}  F1={f1:.3f}")